<a href="https://colab.research.google.com/github/Osujan/MLTSA25_SShrestha/blob/main/Inclass/Transformers_TimeSeriesClassificationWithKerasTransformers_ClassCopy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import keras
from keras import layers

In [5]:
def readucr(filename):
    data = np.loadtxt(filename, delimiter="\t")
    y = data[:, 0]
    x = data[:, 1:]
    return x, y.astype(int)


root_url = "https://raw.githubusercontent.com/hfawaz/cd-diagram/master/FordA/"

x_train, y_train = readucr(root_url + "FordA_TRAIN.tsv")
x_test, y_test = readucr(root_url + "FordA_TEST.tsv")

x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.reshape((x_test.shape[0], x_test.shape[1], 1))

n_classes = len(np.unique(y_train))

idx = np.random.permutation(len(x_train))
x_train = x_train[idx]
y_train = y_train[idx]

y_train[y_train == -1] = 0
y_test[y_test == -1] = 0

input_shape = x_train.shape[1:]

In [8]:
x_train.shape

(3601, 500, 1)

In [7]:
x_test.shape


(1320, 500, 1)

In [9]:
from IPython import get_ipython
from IPython.display import display
# %%
import numpy as np
import keras
from keras import layers
# %%
def readucr(filename):
    data = np.loadtxt(filename, delimiter="\t")
    y = data[:, 0]
    x = data[:, 1:]
    return x, y.astype(int)


root_url = "https://raw.githubusercontent.com/hfawaz/cd-diagram/master/FordA/"

x_train, y_train = readucr(root_url + "FordA_TRAIN.tsv")
x_test, y_test = readucr(root_url + "FordA_TEST.tsv")

x_train = x_train.reshape((x_train.shape[0], x_train.shape[1], 1))
x_test = x_test.reshape((x_test.shape[0], x_test.shape[1], 1))

n_classes = len(np.unique(y_train))

idx = np.random.permutation(len(x_train))
x_train = x_train[idx]
y_train = y_train[idx]

y_train[y_train == -1] = 0
y_test[y_test == -1] = 0

input_shape = x_train.shape[1:] # Define input_shape here

# %%
def train_and_evaluate(model, Xtrn, Xtst, Ytrn, Ytst):
    cb_early = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
    model.fit(
        Xtrn,
        Ytrn,
        epochs=100,
        batch_size=32,
        validation_split=0.2,
        callbacks=[cb_early],

    )
    loss_tst, acc_tst = model.evaluate(Xtst, Ytst, verbose=1)
    print(f"Loss calculated on the testing set: {loss_tst:.4f}")
    print(f"Accuracy calculated on the testing set: {acc_tst:.4f}")
    return None
# %%
def transformer_block(
    x,
    input_shape,
    heads=4,
    key_dim=256,
    encoder_proj_dim=4,
    dropout_attention=.25,
    dropout_projection=.25,
):
    # Indented block of code within the function
    x0 = layers.MultiHeadAttention(num_heads=heads, key_dim=key_dim)(x, x)
    x0 = layers.Dropout(dropout_attention)(x0)
    x0 = layers.Add()([x, x0]) #the "residual connection"
    x1 = layers.LayerNormalization(epsilon=1e-6)(x0)

    x1 = layers.Conv1D(filters=encoder_proj_dim, kernel_size=1, activation="relu")(x1)
    x1 = layers.Conv1D(filters=input_shape[1], kernel_size=1, activation="relu")(x1)
    x1 = layers.Dropout(dropout_projection)(x1)
    x1 = layers.Add()([x0, x1])
    x1 = layers.LayerNormalization(epsilon=1e-6)(x1)

    return x1


def build_transformer_model(
    input_shape,
    encoder_blocks=0, # Changed ... to 1 for demonstration purposes. Set a suitable value.
    feed_forward_units=[128],
):
    inputs = keras.Input(shape=input_shape)
    x = inputs

    for _ in range(encoder_blocks):
        x = transformer_block(x, input_shape)

    x = layers.GlobalMaxPooling1D(data_format="channels_first")(x) # Corrected typo "Golobal" to "Global"
    for n in feed_forward_units:
        x = layers.Dense(n, activation="relu")(x)
        x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(2, activation="softmax")(x)
    model = keras.Model(inputs, outputs)

    return model


transformer_model = build_transformer_model(input_shape) # Now input_shape is accessible
transformer_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["sparse_categorical_accuracy"],
)
transformer_model.summary()

train_and_evaluate(transformer_model, x_train, x_test, y_train, y_test)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 500, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 500)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        64,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 64,386 (251.51 KB)

 Trainable params: 64,386 (251.51 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 1.0502 - sparse_categorical_accuracy: 0.5049 - val_loss: 0.7160 - val_sparse_categorical_accuracy: 0.5867
Epoch 2/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8558 - sparse_categorical_accuracy: 0.5701 - val_loss: 0.6524 - val_sparse_categorical_accuracy: 0.6366
Epoch 3/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7707 - sparse_categorical_accuracy: 0.6020 - val_loss: 0.6172 - val_sparse_categorical_accuracy: 0.6699
Epoch 4/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.7058 - sparse_categorical_accuracy: 0.6480 - val_loss: 0.5799 - val_sparse_categorical_accuracy: 0.6865
Epoch 5/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6505 - sparse_categorical_accuracy: 0.6778 - val_loss: 0.5591 - val_sparse_categorical_accuracy: 0.6976
Epoch 6/100
90/90 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6225 - sparse_categorical_accuracy: 0.6753 - val_loss: 0.5443 - val_sparse_categorical_accuracy: 0.7143
Epoch 7/10

NameError: name 'input_shape' is not defined

In [ ]:
def build_ff_model(
    input_shape,
    feed_forward_units=[128]
):
    inputs = ...
    x = inputs

    x = ...
    for ... in ...:
        x = ...
        x = ...

    outputs = ...
    model = keras.Model(..., ...)

    return model


input_shape = x_train.shape[1:]
ff_model = build_ff_model(input_shape)
ff_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=...),
    metrics=["sparse_categorical_accuracy"],
)
ff_model.summary()

train_and_evaluate(ff_model, x_train, x_test, y_train, y_test)